# Hanoi & HCMC, AlphaEarth embedding, GHSL — zero-shot transfer using Milan's SPLIT-trained model

Applies the Milan SPLIT-trained (TRAIN-points-only) embedding model, against GHSL, directly to Hanoi and HCMC, with no local retraining. Fills the gap in the maps inventory: only the CLMS split-mode zero-shot existed before (notebook 02); this is its GHSL counterpart, matching the already-existing GHSL all-points zero-shot.

## Cell 1 · Imports & configuration

In [1]:
import ee
from gee_init import init_gee

init_gee()

import json
import os
import time
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
from shapely.geometry import box

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
UTM_CRS = 'EPSG:32648'
DRIVE_FOLDER = 'GEE_exports'
CITIES = {'Hanoi': (105.85, 21.03), 'HCMC': (106.70, 10.78)}
LABEL_COL = 'GHSL'

GEE initialised (project: imss21).


---
## Cell 2 · Load Milan's split-trained (TRAIN-only) model + data, build the GEE classifier

In [2]:
with open('./output/milan/ghsl/embedding/model_metadata_spatialcv.json') as f:
    milan_meta = json.load(f)
band_names = milan_meta['band_names']
best_params = milan_meta['best_params']
print(f'Milan split-trained (RF) model: {len(band_names)} bands')
print(f'best_params: {best_params}')

milan_train = gpd.read_file('./output/milan/ghsl/embedding/spatial_train_pts.gpkg').dropna(subset=band_names).reset_index(drop=True)
print(f'Milan TRAIN points (split mode): {len(milan_train)}')

milan_features = [ee.Feature(None, {**{b: float(row[b]) for b in band_names}, LABEL_COL: float(row[LABEL_COL])})
                   for _, row in milan_train.iterrows()]
milan_fc = ee.FeatureCollection(milan_features)

mf = str(best_params.get('max_features', 'sqrt'))
if mf == 'sqrt':
    vps = max(1, int(np.sqrt(len(band_names))))
elif mf == 'log2':
    vps = max(1, int(np.log2(len(band_names))))
else:
    try:
        vps = max(1, int(float(mf) * len(band_names)))
    except (TypeError, ValueError):
        vps = 8

try:
    bag = float(best_params.get('max_samples') or 0.8)
except (TypeError, ValueError):
    bag = 0.8

milan_rf_gee = (
    ee.Classifier.smileRandomForest(
        numberOfTrees=int(float(best_params.get('n_estimators', 300))),
        variablesPerSplit=vps,
        minLeafPopulation=int(float(best_params.get('min_samples_leaf', 2))),
        bagFraction=bag,
        seed=RANDOM_STATE)
    .setOutputMode('REGRESSION')
    .train(features=milan_fc, classProperty=LABEL_COL, inputProperties=band_names))
print('GEE classifier trained on Milan split (TRAIN-only) GHSL data.')

Milan split-trained (RF) model: 64 bands
best_params: {'bootstrap': 'True', 'ccp_alpha': '0.0', 'criterion': 'squared_error', 'max_depth': 'None', 'max_features': '0.3', 'max_leaf_nodes': 'None', 'max_samples': '0.9', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '2', 'min_samples_split': '2', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_estimators': '300', 'n_jobs': '1', 'oob_score': 'False', 'random_state': '42', 'verbose': '0', 'warm_start': 'False'}


Milan TRAIN points (split mode): 2450


GEE classifier trained on Milan split (TRAIN-only) GHSL data.


---
## Cell 3 · Classify each city's own composite, export

In [3]:
for CITY, (lon, lat) in CITIES.items():
    print(f'\n=== {CITY} ===')
    centre_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy([lon], [lat]), crs='EPSG:4326').to_crs(UTM_CRS)
    cx, cy = centre_gdf.geometry.x[0], centre_gdf.geometry.y[0]
    bbox_utm  = box(cx - 15000, cy - 15000, cx + 15000, cy + 15000)
    bbox_4326 = gpd.GeoSeries([bbox_utm], crs=UTM_CRS).to_crs('EPSG:4326').iloc[0]
    aoi_geom  = ee.Geometry.Polygon([[list(c) for c in bbox_4326.exterior.coords]])

    non_water_mask = (ee.ImageCollection('ESA/WorldCover/v200')
                      .filter(ee.Filter.bounds(aoi_geom)).mosaic().clip(aoi_geom)).neq(80)

    start_date = ee.Date.fromYMD(2018, 1, 1)
    aef_col = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
               .filter(ee.Filter.date(start_date, start_date.advance(1, 'year')))
               .filter(ee.Filter.bounds(aoi_geom)))
    aef_proj = aef_col.first().projection()
    city_img = (aef_col.mosaic().setDefaultProjection(aef_proj)
                .clip(aoi_geom).updateMask(non_water_mask))

    pred_img = city_img.classify(milan_rf_gee).rename('IMD_predicted').clamp(0, 100)

    OUTPUT_DIR = f'./output/transfer_vietnam/{CITY.lower()}/embedding_zeroshot_split'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fname = f'{LABEL_COL}_{CITY}_10m_zeroshot_split_embedding'
    out_tif = f'{OUTPUT_DIR}/{fname}.tif'
    if os.path.exists(out_tif):
        print(f'  raster already present -- skipping export.')
        continue

    task = ee.batch.Export.image.toDrive(
        image=pred_img, description=fname, folder=DRIVE_FOLDER,
        fileNamePrefix=fname, region=aoi_geom, scale=10, crs=UTM_CRS,
        maxPixels=1e10, fileFormat='GeoTIFF')
    task.start()
    print(f'  submitted batch export task (id={task.id})')
    state = 'READY'
    while state not in ('COMPLETED', 'FAILED', 'CANCELLED'):
        time.sleep(15)
        state = task.status()['state']
        print(f'    {state}')
    if state != 'COMPLETED':
        raise RuntimeError(f'export {state} -- {task.status()}')
    creds = ee.data.get_persistent_credentials()
    drive = gbuild('drive', 'v3', credentials=creds)
    resp = drive.files().list(q=f"name = '{fname}.tif' and trashed = false",
                               fields='files(id, createdTime)').execute()
    files = sorted(resp.get('files', []), key=lambda f: f['createdTime'], reverse=True)
    if not files:
        raise RuntimeError(f'export completed but {fname}.tif not found on Drive.')
    request = drive.files().get_media(fileId=files[0]['id'])
    with open(out_tif, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(f'  downloaded -> {out_tif}')

    meta = {'band_names': band_names, 'best_params': best_params, 'n_milan_points': len(milan_train),
            'source': 'Milan split-trained (TRAIN-only, embedding) GHSL model, zero-shot transfer -- no local retrain'}
    with open(f'{OUTPUT_DIR}/model_metadata_zeroshot_split.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)

print('\nDone.')


=== Hanoi ===


  submitted batch export task (id=USO3FE6PZVAXYRTHCO4D7RY6)


    RUNNING


    RUNNING


    RUNNING


...

    RUNNING


    RUNNING


    RUNNING


    COMPLETED


  downloaded -> ./output/transfer_vietnam/hcmc/embedding_zeroshot_split/GHSL_HCMC_10m_zeroshot_split_embedding.tif

Done.
